# Notebook 2 — Modelo Prophet

**Proyecto:** Sistema de pronóstico de demanda eléctrica en Ecuador

Implementación del modelo Prophet (Meta), con descomposición automática de tendencia y estacionalidad. Se evalúan los modos aditivo y multiplicativo, y se hace tuning de hiperparámetros sobre el conjunto de validación.

**Pasos:**
1. Cargar `train.csv`, `val.csv`, `test.csv`
2. Reformatear al formato esperado por Prophet (columnas `ds` y `y`)
3. Tuning de hiperparámetros con búsqueda en grilla
4. Ajuste del mejor modelo sobre train+val
5. Predicción sobre test
6. Métricas y exportación

**Requisito:** ejecutar primero Notebook 0.


## 1. Instalación e importaciones

In [ ]:
# En Colab, Prophet viene preinstalado en la mayoría de los runtimes.
# Si falla la importación, descomenta la línea de instalación:
# !pip install prophet --quiet

import warnings
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
import logging
logging.getLogger("prophet").setLevel(logging.WARNING)
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["font.family"] = "serif"

def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def reportar(y_true, y_pred, etiqueta=""):
    mae_v  = mean_absolute_error(y_true, y_pred)
    rmse_v = rmse(y_true, y_pred)
    mape_v = mape(y_true, y_pred)
    print(f"{etiqueta:20s}  MAE={mae_v:8.2f}  RMSE={rmse_v:8.2f}  MAPE={mape_v:6.2f}%")
    return {"MAE": mae_v, "RMSE": rmse_v, "MAPE": mape_v}


## 2. Carga y preparación de datos

Prophet espera un DataFrame con dos columnas: `ds` (fecha) y `y` (valor).


In [ ]:
train_csv = pd.read_csv("train.csv", parse_dates=["fecha"])
val_csv   = pd.read_csv("val.csv",   parse_dates=["fecha"])
test_csv  = pd.read_csv("test.csv",  parse_dates=["fecha"])

def to_prophet(df):
    return df.rename(columns={"fecha": "ds", "demanda_gwh": "y"})

train = to_prophet(train_csv)
val   = to_prophet(val_csv)
test  = to_prophet(test_csv)

train_val = pd.concat([train, val]).reset_index(drop=True)

print(f"Train: {len(train)} obs")
print(f"Val:   {len(val)} obs")
print(f"Test:  {len(test)} obs")


## 3. Tuning de hiperparámetros sobre validación

Espacio de búsqueda:
- `seasonality_mode`: aditivo o multiplicativo
- `changepoint_prior_scale`: 0.001, 0.01, 0.1, 0.5
- `seasonality_prior_scale`: 0.1, 1.0, 10.0


In [ ]:
seasonality_modes      = ["additive", "multiplicative"]
changepoint_priors     = [0.001, 0.01, 0.1, 0.5]
seasonality_priors     = [0.1, 1.0, 10.0]

resultados = []
combos = list(itertools.product(seasonality_modes, changepoint_priors, seasonality_priors))
print(f"Total de combinaciones a evaluar: {len(combos)}")

for seas_mode, cp_prior, seas_prior in combos:
    try:
        m = Prophet(
            seasonality_mode=seas_mode,
            changepoint_prior_scale=cp_prior,
            seasonality_prior_scale=seas_prior,
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False,
        )
        m.fit(train)
        future = m.make_future_dataframe(periods=len(val), freq="MS")
        forecast = m.predict(future)
        pred_val = forecast.iloc[-len(val):]["yhat"].values
        mae_v = mean_absolute_error(val["y"].values, pred_val)
        resultados.append({
            "seasonality_mode": seas_mode,
            "changepoint_prior": cp_prior,
            "seasonality_prior": seas_prior,
            "MAE_val": mae_v
        })
    except Exception as e:
        continue

df_grid = pd.DataFrame(resultados).sort_values("MAE_val").reset_index(drop=True)
print("\nTOP 10 combinaciones por MAE sobre validación:")
print(df_grid.head(10).to_string(index=False))


## 4. Ajuste del mejor modelo sobre train+val

In [ ]:
mejor = df_grid.iloc[0]
print(f"Mejor configuración:")
print(f"  seasonality_mode        = {mejor.seasonality_mode}")
print(f"  changepoint_prior_scale = {mejor.changepoint_prior}")
print(f"  seasonality_prior_scale = {mejor.seasonality_prior}")

modelo = Prophet(
    seasonality_mode=mejor.seasonality_mode,
    changepoint_prior_scale=mejor.changepoint_prior,
    seasonality_prior_scale=mejor.seasonality_prior,
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
)
modelo.fit(train_val)


## 5. Predicción sobre el conjunto de prueba

In [ ]:
future = modelo.make_future_dataframe(periods=len(test), freq="MS")
forecast = modelo.predict(future)

# Aislar predicciones del test
forecast_test = forecast.iloc[-len(test):].copy().reset_index(drop=True)
forecast_test["y_real"] = test["y"].values

print("Predicciones sobre TEST:")
comp = forecast_test[["ds", "y_real", "yhat", "yhat_lower", "yhat_upper"]].copy()
comp["error"] = comp["y_real"] - comp["yhat"]
comp["error_pct"] = (comp["error"] / comp["y_real"]) * 100
print(comp.round(2).to_string(index=False))


## 6. Cálculo de métricas

In [ ]:
# In-sample
forecast_in = forecast.iloc[:len(train_val)]
print("Desempeño del modelo Prophet:\n")
m_train = reportar(train_val["y"].values, forecast_in["yhat"].values, "Train+Val (in-sample)")
m_test  = reportar(test["y"].values,      forecast_test["yhat"].values, "Test (out-of-sample)")


## 7. Componentes de Prophet (tendencia + estacionalidad)

In [ ]:
fig = modelo.plot_components(forecast)
fig.set_size_inches(13, 7)
plt.tight_layout()
plt.savefig("fig_prophet_componentes.png", dpi=300, bbox_inches="tight")
plt.show()


## 8. Gráfico de predicción

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(train["ds"], train["y"], label="Train", color="#1f4e79", lw=1.5)
ax.plot(val["ds"],   val["y"],   label="Val",   color="#2e7d32", lw=1.5)
ax.plot(test["ds"],  test["y"],  label="Test (real)", color="#c62828", lw=1.8, marker="o", ms=5)
ax.plot(forecast_test["ds"], forecast_test["yhat"],
        label="Prophet (pred.)", color="#ef6c00", lw=2.0, marker="s", ms=5, linestyle="--")
ax.fill_between(forecast_test["ds"], forecast_test["yhat_lower"], forecast_test["yhat_upper"],
                color="#ef6c00", alpha=0.18, label="IC 80%")
ax.set_title("Prophet: predicción del conjunto de prueba")
ax.set_ylabel("Demanda (GWh)"); ax.set_xlabel("Fecha"); ax.legend(loc="lower left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("fig_prophet_prediccion.png", dpi=300, bbox_inches="tight")
plt.show()


## 9. Exportación de resultados

In [ ]:
resultados_prophet = pd.DataFrame({
    "fecha": test["ds"].values,
    "real_gwh": test["y"].values,
    "prophet_pred_gwh": forecast_test["yhat"].values,
    "prophet_ic_lower": forecast_test["yhat_lower"].values,
    "prophet_ic_upper": forecast_test["yhat_upper"].values,
})
resultados_prophet.to_csv("resultados_prophet.csv", index=False)

metricas_prophet = pd.DataFrame([
    {"modelo": "Prophet", "conjunto": "train+val", **m_train},
    {"modelo": "Prophet", "conjunto": "test",      **m_test},
])
metricas_prophet.to_csv("metricas_prophet.csv", index=False)

import json
config = {
    "modelo": "Prophet",
    "seasonality_mode": str(mejor.seasonality_mode),
    "changepoint_prior_scale": float(mejor.changepoint_prior),
    "seasonality_prior_scale": float(mejor.seasonality_prior),
    "yearly_seasonality": True,
    "MAE_val": float(mejor.MAE_val)
}
with open("config_prophet.json", "w") as f:
    json.dump(config, f, indent=2)

print("Exportado:")
print("  resultados_prophet.csv")
print("  metricas_prophet.csv")
print("  config_prophet.json")
print("  fig_prophet_prediccion.png, fig_prophet_componentes.png")
print("\nNotebook 2 completado. Continúa con Notebook 3 (LSTM).")
